# Diplomatura en Python para la Ciencia de Datos y Machine Learning
## Trabajo Práctico 2: Perceptrón

**Tema:** Perceptrón y Clasificación sobre el conjunto de datos MNIST.

La base de datos del **MNIST** (Modified National Institute of Standards and Technology) contiene imágenes de 28X28 píxeles en blanco y negro, divididas en 60.000 imágenes de entrenamiento y 10.000 imágenes de prueba.

### 1. Carga del Conjunto de Datos y Exploración Inicial

Ejecuta el código necesario para importar el conjunto de datos MNIST.

**Ejercicio 1:** 
1. Completa las variables necesarias para cargar el conjunto de entrenamiento y de prueba (`X_train`, `y_train`, `X_test`, `y_test`).
2. Responde: ¿Qué forma (`shape`) tienen los conjuntos de datos de entrenamiento y de prueba? Observa e imprime el primer elemento de cada conjunto.

In [ ]:
# Importar el conjunto de datos y guardar en variables
from tensorflow.keras.datasets import mnist

# Cargar el dataset mnist.load_data()
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Imprimir las formas (shape) de X_train, y_train, X_test, y_test
print("X_train:", X_train.shape)   # (60000, 28, 28) -> 60000 imagenes de 28x28 pixeles
print("y_train:", y_train.shape)   # (60000,)        -> 60000 etiquetas (0 al 9)
print("X_test :", X_test.shape)    # (10000, 28, 28) -> 10000 imagenes de 28x28 pixeles
print("y_test :", y_test.shape)    # (10000,)        -> 10000 etiquetas (0 al 9)

# Observar el primer elemento de X_train e y_train
print("
Primer elemento de X_train (matriz 28x28 con valores de 0 a 255):")
print(X_train[0])
print("
Primer elemento de y_train (etiqueta real):", y_train[0])

# Y el primer elemento del conjunto de prueba
print("
Primer elemento de y_test (etiqueta real):", y_test[0])


### 2. Visualización de Muestras

Visualizamos las primeras 8 imágenes del conjunto de entrenamiento utilizando `matplotlib` .

In [ ]:
# Visualizar el conjunto de datos
import matplotlib.pyplot as plt

plt.figure(figsize=(28, 28))
for i, img in enumerate(X_train[:8]):
    plt.subplot(1, 8, i + 1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title("Ejemplo " + str(i + 1))
plt.show()


### 3. Preprocesamiento: Aplanado de las Imágenes (Reshape)

`X_train` tiene una forma inicial de `(60000, 28, 28)` [cite: 3]. Para poder alimentar el modelo Perceptrón de `scikit-learn`, se deben aplanar las imágenes de matriz 2D ($28 \times 28$) a un vector 1D ($784$ elementos).

**Ejercicio 2:** 
1. Completa la línea de código para aplanar `X_train` y `X_test` a 2 dimensiones utilizando `.reshape(...)` .
2. Observa e imprime la forma (`shape`) de los nuevos conjuntos aplanados (`X_train_reshaped` y `X_test_reshaped`).

In [ ]:
from sklearn.linear_model import Perceptron

# Aplanar las imagenes de X_train y X_test usando reshape
# Cada imagen de 28x28 pasa a ser un vector de 784 elementos
X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
X_test_reshaped = X_test.reshape(X_test.shape[0], -1)

# Imprimir la forma de los nuevos conjuntos de datos aplanados
print("X_train_reshaped:", X_train_reshaped.shape)   # (60000, 784)
print("X_test_reshaped :", X_test_reshaped.shape)    # (10000, 784)


### 4. Definición y Entrenamiento del Perceptrón

Definimos el clasificador Perceptrón y entrenamos el modelo sobre el conjunto aplanado.
n_jobs=-1: Controla el paralelismo del procesamiento durante el entrenamiento. El valor -1 le indica a Python que utilice todos los núcleos de procesamiento disponibles en la CPU, lo que acelera significativamente el ajuste del modelo en tareas multiclase como MNIST

In [ ]:
# Definir el modelo del perceptron
modelo = Perceptron(max_iter=2000, random_state=40, n_jobs=-1)

# Entrenar el modelo con los datos aplanados
modelo.fit(X_train_reshaped, y_train)


**Ejercicio 3:** 
1. Ejecuta `modelo.coef_.shape` para conocer el número de parámetros (pesos) del modelo.
2. Responde: ¿Cuántas clases y cuántos pesos tenemos en total luego de ejecutar este comando ?

In [ ]:
# Consultar el numero de parametros que forman el modelo
print("Forma de los coeficientes:", modelo.coef_.shape)   # (10, 784)

n_clases, n_pesos_por_clase = modelo.coef_.shape
print("Clases:", n_clases)                                 # 10 clases (digitos del 0 al 9)
print("Pesos por clase:", n_pesos_por_clase)               # 784 pesos (uno por pixel)
print("Pesos totales:", modelo.coef_.size)                 # 10 * 784 = 7840 pesos

# Respuesta: el modelo tiene 10 clases (un perceptron por digito, estrategia One-vs-Rest)
# y 784 pesos por clase, es decir 7840 pesos en total, mas 10 bias (uno por clase),
# lo que da 7850 parametros entrenables.


**Ejercicio 4:** 
1. Consulta el atributo `modelo.intercept_` para obtener los sesgos / ordenadas al origen (*bias*) .
2. Lista y responde sobre el conjunto de los *bias* obtenido.

In [ ]:
# Consultar los parametros bias / intercept de cada clase
print("Forma del intercept:", modelo.intercept_.shape)   # (10,)
print("Bias por clase:", modelo.intercept_)

for clase, bias in zip(modelo.classes_, modelo.intercept_):
    print(f"Clase {clase}: bias = {bias}")

# Respuesta: hay 10 bias, uno por cada clase (digitos 0 a 9). Cada bias es la ordenada
# al origen del perceptron de esa clase: desplaza el hiperplano de decision para que no
# tenga que pasar obligatoriamente por el origen.


### 5. Predicción y Evaluación

**Ejercicio 5:** 
1. Utiliza `modelo.predict(...)` para realizar la predicción sobre el conjunto de prueba (`X_test_reshaped`).
2. Compara la primera predicción realizada con el primer valor real de `y_test`.

In [ ]:
# Realizar la prediccion con el conjunto de datos de prueba
y_pred = modelo.predict(X_test_reshaped)

# Comparar el primer elemento de la prediccion con el primer elemento real de y_test
print("Primera prediccion:", y_pred[0])
print("Primer valor real :", y_test[0])
print("Coinciden?:", y_pred[0] == y_test[0])

# Comparacion de los primeros 10 casos
print("
Predicciones:", y_pred[:10])
print("Reales      :", y_test[:10])


#### Cálculo de la Puntuación F1 ($F_1$-score)

La **puntuación $F_1$** ($F_1$-score) es la media armónica entre la **precisión** (*precision*) y la **exhaustividad** (*recall*), siendo una métrica estándar para evaluar la calidad global de la clasificación.

**Ejercicio 6:** Completa el código para importar `f1_score` desde `sklearn.metrics` y calcula el $F_1$-score promedio ponderado (`average="weighted"`).

In [ ]:
# Importar f1_score desde sklearn.metrics
from sklearn.metrics import f1_score

# Calcular y mostrar el f1_score ponderado entre y_test e y_pred
f1 = f1_score(y_test, y_pred, average="weighted")
print("F1-score (weighted):", f1)
print(f"F1-score (weighted): {f1:.4f}  ->  {f1 * 100:.2f}%")


### 6. Análisis de Errores de Clasificación

**Ejercicio 7:** 
El siguiente código busca identificar los índices de las imágenes mal clasificadas por el Perceptrón [cite: 3].
Completa las partes incompletas `...` para:
1. Recorrer en paralelo las etiquetas reales (`y_test`) y las predichas (`y_pred`) [cite: 3].
2. Guardar en `index_errors` los índices donde `label != predict` [cite: 3].
3. Graficar un subconjunto de estas imágenes mal clasificadas junto con su etiqueta real y predicha [cite: 3].

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Identificar indices donde ocurrio un error en la clasificacion
index = 0
index_errors = []

# Recorrer en paralelo las etiquetas reales y las predichas
for label, predict in zip(y_test, y_pred):
    if label != predict:
        index_errors.append(index)
    index += 1

print(f"Total de imagenes mal clasificadas: {len(index_errors)}")
print(f"Total de imagenes evaluadas: {len(y_test)}")
print(f"Porcentaje de error: {len(index_errors) / len(y_test) * 100:.2f}%")


In [ ]:
# Graficar algunas imagenes mal clasificadas (ejemplo: indices 8 al 15)
plt.figure(figsize=(20, 4))
for i, img_index in zip(range(1, 9), index_errors[8:16]):
    plt.subplot(1, 8, i)
    # Redimensionar la imagen a 28x28 e imprimirla con su valor real y predicho
    plt.imshow(np.reshape(X_test[img_index], (28, 28)), cmap=plt.cm.gray)
    plt.title('Orig:' + str(y_test[img_index]) + ' Pred:' + str(y_pred[img_index]))
    plt.axis('off')
plt.show()
